In [1]:
import re
from datetime import datetime, timedelta

from selenium.webdriver.common.by import By

from src.flights.models.models import Airport, SingleSearch
from src.flights.scrapers.scrapers import OneWayScraper

In [ ]:
search_item = SingleSearch(
    origin=Airport("IST"),
    destination=Airport("ESB"),
    departure_date=datetime.today().date() + timedelta(days=10),
    direct_only=True,
)
scraper = OneWayScraper(search_item)
print(scraper.url)

elements = scraper.get_raw_flight_results()
elements

https://www.google.com/travel/flights?q=Flights%20to%20FCO%20Airport%20from%20FMM%20on%202024-12-14%20oneway%20direct&curr=EUR&gl=IT
https://www.google.com/travel/flights/search?tfs=CBwQAhogEgoyMDI0LTEyLTE0KABqBwgBEgNGTU1yBwgBEgNGQ09AAUgBcAGCAQsI____________AZgBAg&tfu=EgoIABAAGAAgAigB&gl=IT&curr=EUR


[<selenium.webdriver.remote.webelement.WebElement (session="e2cfbd0df87c6deb6f872702a2632efe", element="f.56D88B07D3445C562132BC0208AF753F.d.1D3FBD64271A2582E8198B56DC976873.e.72")>]

In [3]:
e1 = elements[0]
e1

<selenium.webdriver.remote.webelement.WebElement (session="e2cfbd0df87c6deb6f872702a2632efe", element="f.56D88B07D3445C562132BC0208AF753F.d.1D3FBD64271A2582E8198B56DC976873.e.72")>

In [4]:
items = e1.find_elements(By.TAG_NAME, "li")
items

[<selenium.webdriver.remote.webelement.WebElement (session="e2cfbd0df87c6deb6f872702a2632efe", element="f.56D88B07D3445C562132BC0208AF753F.d.1D3FBD64271A2582E8198B56DC976873.e.73")>]

In [26]:
items[0].text

'6:10\u202fPM\n – \n7:15\u202fPM\nAJetOperated by Turkish Airlines\n1 hr 5 min\nSAW–ESB\nNonstop\n50 kg CO2e\n-25% emissions\n€20'

In [5]:
# AIRLINE LOGO
data = items[0].find_element(By.CLASS_NAME, "EbY4Pc").get_attribute("style")
match = re.search(r"url\((.*?)\)", data)

airline_logo = match.group(1)
airline_logo

'https://www.gstatic.com/flights/airline_logos/70px/FR.png'

In [6]:
# DEPARTURE & ARRIVAL TIME
# TODO: handle +- X days and add date to the time (datetime object)

data = items[0].find_element(By.CLASS_NAME, "mv1WYe").text

dep_time, arr_time = data.replace("\n", "").replace("\u202f", "").split(" – ")

dep_time, arr_time

('7:45AM', '9:15AM')

In [7]:
# AIRLINE

airline = items[0].find_element(By.CSS_SELECTOR, ".sSHqwe.tPgKwe.ogfYpf").text
airline

'RyanairOperated by Malta Air'

In [8]:
# FLIGHT TIME

flight_time = items[0].find_element(By.CSS_SELECTOR, ".gvkrdb.AdWm1c.tPgKwe.ogfYpf").text
flight_time

'1 hr 30 min'

In [9]:
# AIRPORT CODES

airport_codes = items[0].find_element(By.CSS_SELECTOR, ".PTuQse.sSHqwe.tPgKwe.ogfYpf").text
airport_codes = tuple(airport_codes.split("–"))
airport_codes

('FMM', 'FCO')

In [10]:
# STOPS
stops = items[0].find_element(By.CLASS_NAME, "BbR8Ec").text
stops

'Nonstop'

In [11]:
from selenium.common.exceptions import NoSuchElementException

# ONLY HAND LUGGAGE
# TODO: test with a flight that has only hand luggage
try:
    only_hand_luggage = items[0].find_element(By.CSS_SELECTOR, ".vmWDCc.NMm5M") is not None
except NoSuchElementException:
    only_hand_luggage = False
only_hand_luggage

True